In [1]:
import pandas as pd
import numpy as np
import os

train_raw = pd.read_csv('../data/train.csv')
test_raw = pd.read_csv('../data/test.csv')

# Guardar PassengerId de test para el archivo de envío final
passenger_ids_test = test_raw['PassengerId']

# Unir ambos datasets (test no tiene 'Survived', quedará como NaN)
all_df = pd.concat([train_raw, test_raw], sort=False).reset_index(drop=True)

print(f"Filas totales: {all_df.shape[0]} (train: {train_raw.shape[0]}, test: {test_raw.shape[0]})")
all_df.head()

Filas totales: 1309 (train: 891, test: 418)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# Extraer el Deck (primera letra de la cabina) ANTES de perder esa información
df['Deck'] = df['Cabin'].str[0]
df['Deck'] = df['Deck'].fillna('Unknown')

# Mantener también HasCabin (puede seguir siendo útil como variable adicional)
df['HasCabin'] = df['Cabin'].notnull().astype(int)

# Ahora sí, eliminar la columna original
df = df.drop(columns=['Cabin'])

df['Deck'].value_counts()

NameError: name 'df' is not defined

In [ ]:
mediana_age_train = train_raw.groupby(['Pclass', 'Sex'])['Age'].median()

def imputar_age(row, medianas):
    if pd.isnull(row['Age']):
        return medianas.loc[row['Pclass'], row['Sex']]
    return row['Age']

train_raw['Age'] = train_raw.apply(lambda row: imputar_age(row, mediana_age_train), axis=1)
test_raw['Age'] = test_raw.apply(lambda row: imputar_age(row, mediana_age_train), axis=1)

print(f"Nulos en train: {train_raw['Age'].isnull().sum()}, en test: {test_raw['Age'].isnull().sum()}")

Nulos en train: 0, en test: 0


In [ ]:
moda_embarked_train = train_raw['Embarked'].mode()[0]
mediana_fare_train = train_raw['Fare'].median()

train_raw['Embarked'] = train_raw['Embarked'].fillna(moda_embarked_train)
test_raw['Embarked'] = test_raw['Embarked'].fillna(moda_embarked_train)

train_raw['Fare'] = train_raw['Fare'].fillna(mediana_fare_train)
test_raw['Fare'] = test_raw['Fare'].fillna(mediana_fare_train)

np.int64(0)

In [ ]:
for df in [train_raw, test_raw]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

np.int64(0)

In [ ]:
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Capt': 'Rare', 'Sir': 'Rare'
}

for df in [train_raw, test_raw]:
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    df['Title'] = df['Title'].map(title_map)
    df['Title'] = df['Title'].fillna('Rare')

In [ ]:
# AgeBin usa bins fijos (no depende de datos), es seguro directo
for df in [train_raw, test_raw]:
    df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Niño', 'Adolescente', 'Adulto', 'Adulto_mayor', 'Anciano'])

# FareBin usa qcut (cuantiles), hay que definir los cortes SOLO con train
_, bins_fare = pd.qcut(train_raw['Fare'], 4, labels=['Baja', 'Media', 'Alta', 'MuyAlta'], retbins=True)

train_raw['FareBin'] = pd.qcut(train_raw['Fare'], 4, labels=['Baja', 'Media', 'Alta', 'MuyAlta'])
test_raw['FareBin'] = pd.cut(test_raw['Fare'], bins=bins_fare, labels=['Baja', 'Media', 'Alta', 'MuyAlta'], include_lowest=True)

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [ ]:
def ticket_item(x):
    items = x.split(" ")
    if len(items) == 1:
        return "NONE"
    return "_".join(items[0:-1]).replace(".", "").replace("/", "").upper()

train_raw['Ticket_item'] = train_raw['Ticket'].apply(ticket_item)
test_raw['Ticket_item'] = test_raw['Ticket'].apply(ticket_item)

conteo_tickets_train = train_raw['Ticket_item'].value_counts()
categorias_frecuentes = conteo_tickets_train[conteo_tickets_train >= 10].index

train_raw['Ticket_item'] = train_raw['Ticket_item'].apply(lambda x: x if x in categorias_frecuentes else 'RARE')
test_raw['Ticket_item'] = test_raw['Ticket_item'].apply(lambda x: x if x in categorias_frecuentes else 'RARE')

,Age,AgeBin,Fare,FareBin
0,22.0,Adulto,7.2500,Baja
1,38.0,Adulto_mayor,71.2833,MuyAlta
2,26.0,Adulto,7.9250,Media
3,35.0,Adulto,53.1000,MuyAlta
4,35.0,Adulto,8.0500,Media


In [ ]:
for df in [train_raw, test_raw]:
    df.drop(columns=['Name', 'Ticket', 'Age'], inplace=True)

train_raw.drop(columns=['PassengerId'], inplace=True)
test_raw.drop(columns=['PassengerId'], inplace=True)  # ya guardamos passenger_ids_test aparte

train_raw.isnull().sum()

Title
Mr        517
Miss      185
Mrs       125
Master     40
Rare       22
Name: count, dtype: int64

In [ ]:
# Sex: male/female -> 0/1
train_raw['Sex'] = train_raw['Sex'].map({'male': 0, 'female': 1})
test_raw['Sex'] = test_raw['Sex'].map({'male': 0, 'female': 1})

# One-hot encoding por separado
train_encoded = pd.get_dummies(train_raw, columns=['Embarked', 'Title', 'AgeBin', 'FareBin', 'Ticket_item', 'Deck'], drop_first=True)
test_encoded = pd.get_dummies(test_raw, columns=['Embarked', 'Title', 'AgeBin', 'FareBin', 'Ticket_item', 'Deck'], drop_first=True)

# Alinear columnas: test debe tener EXACTAMENTE las mismas columnas que train (sin Survived)
train_cols = train_encoded.drop(columns=['Survived']).columns
test_encoded = test_encoded.reindex(columns=train_cols, fill_value=0)

print(f"Columnas en train (sin Survived): {len(train_cols)}")
print(f"Columnas en test: {len(test_encoded.columns)}")
print(f"¿Coinciden?: {list(train_cols) == list(test_encoded.columns)}")

Ticket_item
NONE          665
PC             60
CA             41
A5             21
SOTONOQ        15
STONO_2        12
SCPARIS        11
WC             10
A4              7
STONO2          6
SOC             6
C               5
FCC             5
PP              3
WEP             3
SOPP            3
SWPP            2
PPP             2
SCAH            2
SOTONO2         2
SCA4            1
SP              1
SOP             1
FA              1
SCOW            1
SC              1
AS              1
SCAH_BASLE      1
FC              1
CASOTON         1
Name: count, dtype: int64

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
train_encoded.to_csv('../data/processed/train_clean.csv', index=False)
test_encoded.to_csv('../data/processed/test_clean.csv', index=False)
print("Archivos guardados correctamente ✅")

Ticket_item
NONE       665
PC          60
RARE        56
CA          41
A5          21
SOTONOQ     15
STONO_2     12
SCPARIS     11
WC          10
Name: count, dtype: int64

In [ ]:
# Name y Ticket son texto libre con poco valor predictivo directo
# PassengerId es solo un identificador
df = df.drop(columns=['Name', 'Ticket', 'PassengerId'])

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Deck,HasCabin,FamilySize,IsAlone,Title,AgeBin,FareBin,Ticket_item
0,0,3,male,22.0,1,0,7.2500,S,Unknown,0,2,0,Mr,Adulto,Baja,A5
1,1,1,female,38.0,1,0,71.2833,C,C,1,2,0,Mrs,Adulto_mayor,MuyAlta,PC
2,1,3,female,26.0,0,0,7.9250,S,Unknown,0,1,1,Miss,Adulto,Media,RARE
3,1,1,female,35.0,1,0,53.1000,S,C,1,2,0,Mrs,Adulto,MuyAlta,NONE
4,0,3,male,35.0,0,0,8.0500,S,Unknown,0,1,1,Mr,Adulto,Media,NONE


In [ ]:
# Rellenar los valores nulos de Title con "Rare" 
df['Title'] = df['Title'].fillna('Rare')


In [ ]:
# drop a "age" column since we have AgeBin
df = df.drop(columns=['Age'])

In [ ]:
df.isnull().sum()

Survived       0
Pclass         0
Sex            0
SibSp          0
Parch          0
Fare           0
Embarked       0
Deck           0
HasCabin       0
FamilySize     0
IsAlone        0
Title          0
AgeBin         0
FareBin        0
Ticket_item    0
dtype: int64

In [ ]:
import os

# Crear la carpeta si no existe
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/train_clean.csv', index=False)
print("Dataset limpio guardado correctamente ✅")

Dataset limpio guardado correctamente ✅


In [ ]:
df.head(5)

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked,Deck,HasCabin,FamilySize,IsAlone,Title,AgeBin,FareBin,Ticket_item
0,0,3,male,1,0,7.2500,S,Unknown,0,2,0,Mr,Adulto,Baja,A5
1,1,1,female,1,0,71.2833,C,C,1,2,0,Mrs,Adulto_mayor,MuyAlta,PC
2,1,3,female,0,0,7.9250,S,Unknown,0,1,1,Miss,Adulto,Media,RARE
3,1,1,female,1,0,53.1000,S,C,1,2,0,Mrs,Adulto,MuyAlta,NONE
4,0,3,male,0,0,8.0500,S,Unknown,0,1,1,Mr,Adulto,Media,NONE
